# Session 6 · Structured output and an eval

**From Prompt to Production · EvisionC**

1. Set **ROUTE** in the first cell to your model route from Session 2 (`claude`, `gemini`, `openai` or `openrouter`).
2. Add your API key in Colab Secrets (key icon, left sidebar, Notebook access on) under the name for your route: `ANTHROPIC_API_KEY`, `GEMINI_API_KEY`, `OPENAI_API_KEY` or `OPENROUTER_API_KEY`.
3. Run the cells from top to bottom (Shift+Enter). **Lumo track:** everything runs as is. **Own case:** the comments say what to swap.

The guided player explains every cell.

In [ ]:
# Cell 1 · setup (the same in every lab)
!git clone -q --depth 1 https://github.com/lrebelfor/lumo-sandbox /content/lumo 2>/dev/null || git -C /content/lumo pull -q
!pip install -q anthropic openai
import sys; sys.path.insert(0, "/content/lumo")
import lumo
from course_llm import LLM

ROUTE = "claude"   # your model route from Session 2: "claude", "openai", "gemini", "openrouter" (or "ollama" outside Colab)
llm = LLM(ROUTE)
print("Connected:", llm)

In [ ]:
# Structured output: the answer must match this shape (a JSON schema). The labels come from Lumo's data.
INTENTS = sorted({e["intent"] for e in lumo.emails()})
schema = {
    "type": "object",
    "properties": {
        "intent": {"type": "string", "enum": INTENTS},
        "urgency": {"type": "string", "enum": ["low", "medium", "high"]},
        "order_number": {"type": "string"},
        "summary": {"type": "string"},
    },
    "required": ["intent", "urgency", "order_number", "summary"],
    "additionalProperties": False,
}

def classify(message):
    return llm.json("Classify this customer email for Lumo's support team. "
                    "Use an empty string if there is no order number.\n\n" + message, schema)

print(INTENTS)
print(classify("Third time my parcel #1055 is late. I want my money back NOW."))

In [ ]:
# Your first eval set: 40 Lumo emails, each labelled by a human (Femke). Own case: your 20 examples.
cases = [(e["text"], e["intent"]) for e in lumo.emails()]

passed, failures = 0, []
for message, expected in cases:
    got = classify(message)["intent"]
    passed += got == expected
    if got != expected:
        failures.append((expected, got, message))
print(f"Score: {passed}/{len(cases)} = {100 * passed / len(cases):.0f}%\n")
for expected, got, message in failures:
    print(f"FAIL | expected {expected:16} got {got:16} | {message[:70]}")